# Iterative Retrieval — Multi-Turn Context Refinement

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/iterative_retrieval.ipynb)

## The Limits of a Single Glance

Imagine researching: *"What is the GDP of the country where the inventor of the telephone was born?"*  A human would break this down — first find *who* invented the telephone, then *where* that person was born, then the GDP of *that* country. Each search refines the next.

Standard RAG performs **one** retrieval pass and feeds whatever it finds to the language model.  For simple factual questions this works — but for **multi-hop** queries the single pass returns incomplete context, forcing the model to hallucinate.

This notebook introduces **Iterative Retrieval**: a strategy that mirrors the human research loop of *search → read → refine → search again*.  Multiple retrieval turns, each guided by what was learned previously, assemble the full chain of evidence needed to answer complex questions.

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

- **Explain** why single-pass retrieval fails for multi-hop and compositional queries
- **Implement** an iterative retrieval loop that refines its search query over multiple rounds
- **Apply** three different query-refinement strategies (LLM rewriting, residual question extraction, gap detection)
- **Design** convergence criteria so the loop knows when to stop
- **Compare** single-pass versus iterative retrieval on a concrete multi-hop question

## 1 — Why Single-Pass Retrieval Fails

Consider the question from the introduction:

> *"What is the GDP of the country where the inventor of the telephone was born?"*

This question contains **three implicit sub-questions** chained together:

| Step | Sub-question | Expected fact |
|------|-------------|---------------|
| 1 | Who invented the telephone? | Alexander Graham Bell |
| 2 | Where was Alexander Graham Bell born? | Edinburgh, Scotland (United Kingdom) |
| 3 | What is the GDP of the United Kingdom? | ≈ $3.1 trillion |

A single embedding-based retrieval pass computes **one** query vector and retrieves the *k* most similar passages.  The trouble is that the query mixes three semantic targets — telephony, biography, and economics — so retrieved passages scatter across topics without covering the full chain.

This is a fundamental **information-theoretic** limitation: a single dense vector cannot faithfully represent a conjunction of independent information needs.  The solution is to *decompose* the retrieval into multiple focused passes.

## 2 — The Iterative Retrieval Loop

### The Researcher Analogy

Think of how an experienced researcher answers a hard question:

1. **Search** — Query a database with the best phrasing you have now.
2. **Read** — Skim the results, extracting useful facts.
3. **Reflect** — *"Do I have enough to answer?  If not, what's missing?"*
4. **Refine** — Formulate a *new* query targeting the missing information.
5. **Repeat** — Go back to step 1.

This **retrieve → read → refine → retrieve** cycle naturally converges: each iteration closes part of the information gap, producing increasingly specific queries and increasingly relevant context.

```
┌─────────────┐
│  Original    │
│   Query      │
└──────┬───────┘
       ▼
┌──────────────┐     ┌────────────────┐
│  Retrieve    │────▶│  Read & Extract │
│  (FAISS)     │     │  key facts      │
└──────────────┘     └───────┬─────────┘
       ▲                     │
       │              ┌──────▼─────────┐
       │              │  Refine Query  │
       └──────────────│  (LLM)        │
                      └────────────────┘
```

The loop terminates when we hit a **convergence criterion** (discussed in Section 5) or a maximum number of iterations.

## 3 — Query Refinement Strategies

Not all refinement strategies are equal.  Here are three progressively sophisticated approaches.

### 3.1 — LLM-Based Query Rewriting

The simplest approach: hand the LLM the original question *and* the retrieved passages, then ask it to produce a **better search query**.  The LLM identifies which part of the question remains unanswered and generates a focused follow-up.

### 3.2 — Residual Question Extraction

Instead of asking for a new *query*, ask the LLM: *"What information is still missing?"*  The answer is a **residual question** — used directly as the next retrieval query.  The advantage is that the LLM explicitly reasons about what it *does not* know, making refinement more targeted.

### 3.3 — Gap Detection

The most structured strategy.  Ask the LLM to produce a **checklist** of facts needed, then compare against facts already retrieved.  The first unchecked item becomes the next query.  This gives the system an explicit *plan*, useful when the number of hops is large.  The downside is extra LLM calls.

In our implementation we use **residual question extraction** (Strategy 3.2) as it strikes the best balance between effectiveness and simplicity.

## 4 — Implementation

### Environment Setup

We need a handful of libraries: `transformers` and `bitsandbytes` for the LLM, `sentence-transformers` for the embedding model, `faiss-cpu` for the vector index, and `datasets` / `accelerate` as supporting dependencies.

In [ ]:
!pip install -q transformers torch sentence-transformers faiss-cpu datasets bitsandbytes accelerate

### Imports and Configuration

In [ ]:
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

### Loading Qwen3-8B (4-bit Quantized)

We load **Qwen3-8B** in 4-bit precision via `bitsandbytes`.  This brings memory from ~16 GB (float16) to roughly 5 GB, feasible on a free Colab T4 GPU.  The model serves as our **query refiner** and **answer generator**.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer.padding_side = "left"
print(f"✓ Loaded {model_name} in 4-bit precision")

### Loading the Embedding Model

For retrieval we use **all-MiniLM-L6-v2**, a compact sentence-transformer that produces 384-dimensional embeddings.  It is fast enough to re-encode queries on every iteration with negligible overhead.

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"✓ Embedding model loaded — dimension: {EMBED_DIM}")

### Building a Knowledge Base

We construct a small but carefully designed knowledge base of 20 passages.  The passages are organized so that answering multi-hop questions requires **chaining facts across multiple passages** — no single passage contains the full answer.

The knowledge base covers four topical "chains":

| Chain | Passages | Target question |
|-------|----------|----------------|
| Telephone → Bell → Scotland → UK GDP | 4 passages | *"GDP of the country where the telephone inventor was born"* |
| Penicillin → Fleming → St Mary's → Imperial College | 4 passages | *"University where penicillin was discovered"* |
| Relativity → Einstein → Ulm → Germany economy | 4 passages | *"Largest economy in the EU and its connection to relativity"* |
| DNA → Watson & Crick → Cavendish → Cambridge | 4 passages | *"Lab where DNA structure was discovered"* |
| General filler passages | 4 passages | — |

In [ ]:
KNOWLEDGE_BASE: list[str] = [
    # ── Chain 1: Telephone → Bell → Scotland → UK GDP ──
    "Alexander Graham Bell is widely credited with inventing the first practical telephone in 1876. "
    "His patent, filed on February 14, 1876, became one of the most valuable patents in history.",

    "Alexander Graham Bell was born on March 3, 1847, in Edinburgh, Scotland. "
    "He grew up in a family deeply involved in the study of elocution and speech, "
    "which influenced his later work on sound transmission.",

    "Scotland is a constituent country of the United Kingdom, located in the northern part of Great Britain. "
    "Edinburgh is its capital city, and Glasgow is the largest city by population.",

    "The United Kingdom has a gross domestic product (GDP) of approximately 3.1 trillion US dollars, "
    "making it the sixth-largest economy in the world as of 2023.",

    # ── Chain 2: Penicillin → Fleming → St Mary's → Imperial College ──
    "Penicillin was discovered by Sir Alexander Fleming in 1928 when he noticed that mold on a "
    "Staphylococcus culture plate had created a bacteria-free circle around itself.",

    "Alexander Fleming made his groundbreaking discovery of penicillin at St Mary's Hospital "
    "in Paddington, London, where he worked as a bacteriologist.",

    "St Mary's Hospital in London is a major teaching hospital and has been part of Imperial College "
    "London since 1988. It is located in the Paddington area of Westminster.",

    "Imperial College London is a world-renowned research university located in South Kensington, London. "
    "It consistently ranks among the top ten universities globally for science and engineering.",

    # ── Chain 3: Relativity → Einstein → Ulm → Germany ──
    "Albert Einstein published his special theory of relativity in 1905, fundamentally changing "
    "our understanding of space, time, and the relationship between energy and mass (E=mc²).",

    "Albert Einstein was born on March 14, 1879, in Ulm, in the Kingdom of Württemberg in the "
    "German Empire. His family moved to Munich shortly after his birth.",

    "Ulm is a city in the German state of Baden-Württemberg, situated on the River Danube. "
    "It is famous for having the tallest church steeple in the world, the Ulm Minster at 161.5 meters.",

    "Germany is the largest economy in the European Union and the fourth-largest in the world, "
    "with a GDP of approximately 4.1 trillion US dollars as of 2023.",

    # ── Chain 4: DNA → Watson & Crick → Cavendish → Cambridge ──
    "The structure of DNA was determined by James Watson and Francis Crick in 1953. "
    "They proposed the double-helix model, one of the most important discoveries in biology.",

    "Watson and Crick conducted their DNA research at the Cavendish Laboratory in Cambridge, England. "
    "Their work built upon X-ray diffraction data from Rosalind Franklin and Maurice Wilkins.",

    "The Cavendish Laboratory is the Department of Physics at the University of Cambridge. "
    "Founded in 1874, it has been the site of numerous Nobel Prize-winning discoveries.",

    "The University of Cambridge, founded in 1209, is one of the oldest and most prestigious "
    "universities in the world. It is located in Cambridge, Cambridgeshire, England.",

    # ── General filler passages ──
    "The speed of light in a vacuum is approximately 299,792,458 meters per second. "
    "This constant, denoted c, is fundamental to the theory of relativity.",

    "The Amazon River is the largest river in the world by discharge volume. "
    "It flows through South America, primarily through Brazil, and empties into the Atlantic Ocean.",

    "The Great Wall of China is a series of fortifications built along the historical northern borders "
    "of China to protect against various nomadic groups. Construction began as early as the 7th century BC.",

    "Marie Curie was the first woman to win a Nobel Prize and remains the only person to win "
    "Nobel Prizes in two different scientific fields: physics (1903) and chemistry (1911).",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} passages")

### Building the FAISS Index

We encode every passage into a dense vector and store them in a FAISS `IndexFlatIP` (inner-product / cosine similarity after normalization).  Because we normalize the embeddings, inner product is equivalent to cosine similarity — and `IndexFlatIP` gives exact results, which is ideal for a small index.

In [ ]:
def build_index(passages: list[str], model: SentenceTransformer) -> faiss.IndexFlatIP:
    """Encode passages and build a FAISS inner-product index."""
    embeddings = model.encode(passages, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype(np.float32))
    return index

faiss_index = build_index(KNOWLEDGE_BASE, embed_model)
print(f"✓ FAISS index built — {faiss_index.ntotal} vectors, dim={EMBED_DIM}")

### Helper Functions

We define two utilities that we will reuse throughout the notebook:

1. **`retrieve`** — Takes a query string, encodes it, and returns the top-*k* most similar passages from the index.
2. **`generate`** — Wraps the LLM call: applies the chat template, tokenizes, calls `model.generate()`, and decodes the response.

Separating these concerns keeps the iterative loop itself clean and readable.

In [ ]:
def retrieve(
    query: str,
    index: faiss.IndexFlatIP,
    passages: list[str],
    embed: SentenceTransformer,
    top_k: int = 3,
) -> list[str]:
    """Retrieve the top-k passages most similar to the query."""
    q_vec = embed.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, idxs = index.search(q_vec, top_k)
    return [passages[i] for i in idxs[0]]

In [ ]:
def generate(prompt: str, max_new_tokens: int = 256) -> str:
    """Send a chat-formatted prompt to Qwen3-8B and return the response."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    # Decode only the newly generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 5 — Query Refinement via Residual Question Extraction

The core of our iterative approach is the **refine** step.  After each retrieval round we ask the LLM:

> *"Given the original question and the facts retrieved so far, what specific information is still missing?  Formulate a short, focused search query to find it."*

This prompt does two things at once: it forces the model to reason about the *gap* between what is known and what is needed, and it produces a ready-to-use follow-up query for the next retrieval round.  The output is typically a single sentence — concise enough to be a good embedding query, yet specific enough to land on the right passages.

In [ ]:
REFINE_PROMPT_TEMPLATE = """You are a research assistant. A user asked the following question:

QUESTION: {question}

So far, the following passages have been retrieved:

{context}

Based on what has been retrieved, determine what information is still missing to fully answer
the question. Write a single, concise search query (one sentence) that would help find the
missing information. Output ONLY the search query, nothing else."""


def refine_query(
    original_question: str,
    retrieved_passages: list[str],
) -> str:
    """Use the LLM to generate a refined follow-up query."""
    context_block = "\n\n".join(
        f"[Passage {i+1}] {p}" for i, p in enumerate(retrieved_passages)
    )
    prompt = REFINE_PROMPT_TEMPLATE.format(
        question=original_question,
        context=context_block,
    )
    return generate(prompt, max_new_tokens=64)

## 6 — Convergence: Knowing When to Stop

An iterative loop needs a **stopping criterion**.  We implement three complementary checks:

1. **Maximum iterations** — A hard upper bound (e.g., 3 rounds).  The safety net.
2. **Query similarity plateau** — If the refined query is too semantically similar to the previous one (cosine > 0.90), further iteration is unlikely to surface new passages.
3. **Passage novelty** — If the new round returns only passages already seen, there is nothing new to learn.

In practice the similarity plateau fires most often — after 2–3 rounds the LLM rephrases the same residual question, signalling that the knowledge base is exhausted for this topic.

In [ ]:
def query_similarity(q1: str, q2: str, embed: SentenceTransformer) -> float:
    """Compute cosine similarity between two query strings."""
    vecs = embed.encode([q1, q2], normalize_embeddings=True)
    return float(np.dot(vecs[0], vecs[1]))


def has_converged(
    prev_query: str,
    new_query: str,
    prev_passages: set[str],
    new_passages: list[str],
    embed: SentenceTransformer,
    sim_threshold: float = 0.90,
) -> tuple[bool, str]:
    """Check if the iterative loop should stop."""
    sim = query_similarity(prev_query, new_query, embed)
    if sim > sim_threshold:
        return True, f"Query similarity plateau ({sim:.3f} > {sim_threshold})"

    new_set = set(new_passages)
    if new_set.issubset(prev_passages):
        return True, "No novel passages retrieved"

    return False, ""  

## 7 — Putting It All Together: The Iterative Retrieval Pipeline

Now we assemble the full pipeline.  The function `iterative_retrieve` runs up to `max_rounds` retrieval iterations, refining the query after each round and accumulating context.  It returns the gathered passages *and* a detailed log of every step, so we can inspect what happened.

In [ ]:
def iterative_retrieve(
    question: str,
    index: faiss.IndexFlatIP,
    passages: list[str],
    embed: SentenceTransformer,
    top_k: int = 3,
    max_rounds: int = 3,
    sim_threshold: float = 0.90,
) -> dict:
    """Run the iterative retrieval loop and return results + log."""
    log: list[dict] = []
    all_passages: list[str] = []
    seen: set[str] = set()
    current_query = question

    for round_num in range(1, max_rounds + 1):
        # ── Retrieve ──
        retrieved = retrieve(current_query, index, passages, embed, top_k=top_k)
        novel = [p for p in retrieved if p not in seen]
        all_passages.extend(novel)
        seen.update(novel)

        round_log = {
            "round": round_num,
            "query": current_query,
            "retrieved_count": len(retrieved),
            "novel_count": len(novel),
        }

        # ── Check convergence ──
        if round_num > 1:
            converged, reason = has_converged(
                prev_query, current_query, prev_seen, retrieved,
                embed, sim_threshold,
            )
            if converged:
                round_log["converged"] = reason
                log.append(round_log)
                print(f"  Round {round_num}: CONVERGED — {reason}")
                break

        log.append(round_log)
        print(f"  Round {round_num}: query='{current_query[:80]}...' "
              f"| retrieved={len(retrieved)} | novel={len(novel)}")

        if round_num == max_rounds:
            break

        # ── Refine query for next round ──
        prev_query = current_query
        prev_seen = set(seen)
        current_query = refine_query(question, all_passages)
        print(f"    ↳ Refined query: '{current_query[:80]}...'")

    return {
        "question": question,
        "passages": all_passages,
        "rounds": len(log),
        "log": log,
    }

## 8 — Demonstration: Single-Pass vs Iterative Retrieval

Let us now test our pipeline on the motivating multi-hop question.  First, we run a **single-pass** retrieval (one round, no refinement) and inspect what comes back.  Then we run the **iterative** version and compare.

### Single-Pass Retrieval

In [ ]:
QUESTION = "What is the GDP of the country where the inventor of the telephone was born?"

print("=" * 70)
print("SINGLE-PASS RETRIEVAL")
print("=" * 70)
single_passages = retrieve(QUESTION, faiss_index, KNOWLEDGE_BASE, embed_model, top_k=3)

for i, p in enumerate(single_passages, 1):
    print(f"\n[Passage {i}] {p[:120]}...")

With a single pass the retriever is likely to find passages about *either* the telephone *or* GDP, but not the connecting facts (Bell's birthplace, Scotland being in the UK).  Let's ask the LLM to answer using only these passages.

In [ ]:
ANSWER_PROMPT = """Answer the following question using ONLY the provided context.
If the context does not contain enough information, say so explicitly.

QUESTION: {question}

CONTEXT:
{context}

ANSWER:"""

single_context = "\n\n".join(single_passages)
single_answer = generate(
    ANSWER_PROMPT.format(question=QUESTION, context=single_context),
    max_new_tokens=200,
)
print("Single-pass answer:")
print(single_answer)

### Iterative Retrieval (3 Rounds)

Now let us run the same question through the iterative pipeline.  Watch the log to see how each round targets a different part of the reasoning chain.

In [ ]:
print("=" * 70)
print("ITERATIVE RETRIEVAL (up to 3 rounds)")
print("=" * 70)

result = iterative_retrieve(
    QUESTION,
    faiss_index,
    KNOWLEDGE_BASE,
    embed_model,
    top_k=3,
    max_rounds=3,
)

In [ ]:
print(f"\nTotal unique passages gathered: {len(result['passages'])}")
print(f"Rounds used: {result['rounds']}\n")

for i, p in enumerate(result["passages"], 1):
    print(f"[Passage {i}] {p[:120]}...")
    print()

In [ ]:
iter_context = "\n\n".join(result["passages"])
iter_answer = generate(
    ANSWER_PROMPT.format(question=QUESTION, context=iter_context),
    max_new_tokens=200,
)
print("Iterative-retrieval answer:")
print(iter_answer)

### Side-by-Side Comparison

Let us print both answers together so the difference is easy to see.

In [ ]:
print("=" * 70)
print("COMPARISON")
print("=" * 70)

print("\n📌 Question:")
print(f"   {QUESTION}")

print("\n--- Single-Pass ---")
print(f"   Passages used: {len(single_passages)}")
print(f"   Answer: {single_answer}")

print("\n--- Iterative (multi-round) ---")
print(f"   Passages used: {len(result['passages'])}")
print(f"   Answer: {iter_answer}")

### What Happened?

In a typical run:

- **Round 1** retrieves passages about the telephone and Alexander Graham Bell.
- **Round 2** uses a refined query like *"Where was Alexander Graham Bell born?"* and finds the Edinburgh/Scotland passage.
- **Round 3** targets the GDP of the United Kingdom, completing the chain.

The single-pass system never gets past the first link because it has no mechanism to *learn from what it found* and *redirect*.  The iterative system builds understanding incrementally — exactly like a human researcher.

## 9 — Second Example: A Different Chain

Let us verify the approach on a second multi-hop question that follows a completely different reasoning chain through our knowledge base.

In [ ]:
QUESTION_2 = "At which university was the laboratory where DNA structure was discovered?"

print("=" * 70)
print("SINGLE-PASS")
print("=" * 70)
sp_passages = retrieve(QUESTION_2, faiss_index, KNOWLEDGE_BASE, embed_model, top_k=3)
for i, p in enumerate(sp_passages, 1):
    print(f"[{i}] {p[:100]}...")

sp_ctx = "\n\n".join(sp_passages)
sp_ans = generate(ANSWER_PROMPT.format(question=QUESTION_2, context=sp_ctx), max_new_tokens=200)
print(f"\nSingle-pass answer: {sp_ans}")

print("\n" + "=" * 70)
print("ITERATIVE")
print("=" * 70)
result2 = iterative_retrieve(
    QUESTION_2, faiss_index, KNOWLEDGE_BASE, embed_model,
    top_k=3, max_rounds=3,
)
ir_ctx = "\n\n".join(result2["passages"])
ir_ans = generate(ANSWER_PROMPT.format(question=QUESTION_2, context=ir_ctx), max_new_tokens=200)
print(f"\nIterative answer: {ir_ans}")

## 10 — Visualizing Convergence

Let us inspect the **query similarity** between successive rounds to see how the system converges.  When the cosine similarity between the current and previous query is high, the LLM is essentially asking the same question again — a clear signal that no new information is available.

In [ ]:
def run_with_similarity_log(
    question: str,
    index: faiss.IndexFlatIP,
    passages: list[str],
    embed: SentenceTransformer,
    top_k: int = 3,
    max_rounds: int = 5,
) -> list[dict]:
    """Run iterative retrieval for more rounds and log query similarities."""
    similarities: list[dict] = []
    all_passages: list[str] = []
    seen: set[str] = set()
    current_query = question

    for round_num in range(1, max_rounds + 1):
        retrieved = retrieve(current_query, index, passages, embed, top_k=top_k)
        novel = [p for p in retrieved if p not in seen]
        all_passages.extend(novel)
        seen.update(novel)

        entry = {"round": round_num, "query": current_query[:60], "novel": len(novel)}

        if round_num > 1:
            sim = query_similarity(prev_query, current_query, embed)
            entry["similarity_to_prev"] = round(sim, 4)
        else:
            entry["similarity_to_prev"] = None

        similarities.append(entry)
        prev_query = current_query

        if round_num < max_rounds:
            current_query = refine_query(question, all_passages)

    return similarities

sim_log = run_with_similarity_log(
    QUESTION, faiss_index, KNOWLEDGE_BASE, embed_model,
    top_k=3, max_rounds=5,
)

print(f"{'Round':<6} {'Sim→Prev':<12} {'Novel':<8} {'Query'}")
print("-" * 80)
for entry in sim_log:
    sim_str = f"{entry['similarity_to_prev']:.4f}" if entry["similarity_to_prev"] else "—"
    print(f"{entry['round']:<6} {sim_str:<12} {entry['novel']:<8} {entry['query']}")

You should observe that the similarity between successive queries **increases** with each round.  After 2–3 rounds the similarity typically exceeds 0.90, confirming that the system has extracted all the information it can from the knowledge base for this question.  This is the **query similarity plateau** that triggers our convergence criterion.

## 11 — Connection to Other Techniques

Iterative retrieval is the **foundation layer** on which several more sophisticated RAG strategies are built.

| Technique | Relationship to Iterative Retrieval |
|-----------|--------------------------------------|
| **Corrective RAG (CRAG)** | Adds a *relevance evaluator* that decides whether to keep, discard, or refine each retrieved passage before the next iteration.  See the [CRAG notebook](crag.ipynb) in this course. |
| **Self-RAG** | Incorporates *self-reflection tokens* that let the model critique its own retrieval and generation quality mid-stream.  See the [Self-RAG notebook](self_rag.ipynb). |
| **Agentic RAG** | Wraps the retrieval loop inside an autonomous agent that can choose *which tool* to call (search, calculator, code interpreter) at each step. |
| **Chain-of-Thought + RAG** | Uses explicit reasoning traces to decompose the question, with each reasoning step triggering a targeted retrieval. |

The key insight is that **all of these techniques share the same retrieve → reason → refine core loop**.  Understanding iterative retrieval gives you the mental model needed to understand every advanced RAG pattern.

## ✏️ Exercises

### Exercise 1 — Implement Gap-Detection Refinement

Replace the residual-question strategy with **gap detection** (Strategy 3.3).  Write a function `refine_query_gap(question, passages)` that:

1. Asks the LLM to list the specific facts needed to answer the question.
2. Asks the LLM which of those facts are already present in the retrieved passages.
3. Returns a search query targeting the first *missing* fact.

Run the iterative pipeline with your new refinement function and compare results against the residual-question approach.

In [ ]:
# Exercise 1 — Starter code

GAP_DETECT_PROMPT = """You are a research assistant. A user asked:

QUESTION: {question}

List exactly the key facts needed to answer this question (one per line, numbered).
Then, for each fact, mark [FOUND] if it appears in the passages below, or [MISSING] if not.

PASSAGES:
{context}

Output format:
1. <fact> — [FOUND/MISSING]
2. <fact> — [FOUND/MISSING]
...

Finally, write a search query to find the first MISSING fact. Put it on the last line
prefixed with QUERY:
"""


def refine_query_gap(original_question: str, retrieved_passages: list[str]) -> str:
    # TODO: Implement gap-detection query refinement
    # 1. Format the prompt using GAP_DETECT_PROMPT
    # 2. Call generate()
    # 3. Parse the output to extract the QUERY: line
    # 4. Return the query string
    raise NotImplementedError("Your implementation here")

### Exercise 2 — Adaptive Top-K

In our current pipeline, `top_k` is fixed at 3 for every round.  But early rounds (broad queries) might benefit from more results, while later rounds (precise queries) need fewer.

Implement an **adaptive top-k** strategy:
- Round 1: `top_k = 5`
- Subsequent rounds: `top_k = max(2, 5 - round_num)`

Modify `iterative_retrieve` to accept a `top_k_schedule` parameter and test it on both example questions.

In [ ]:
# Exercise 2 — Starter code

def adaptive_top_k(round_num: int, initial_k: int = 5) -> int:
    """Return the top-k value for a given round number."""
    # TODO: Implement the adaptive schedule
    raise NotImplementedError("Your implementation here")


# Hint: Modify the iterative_retrieve function to call adaptive_top_k(round_num)
# instead of using a fixed top_k, then run:
#
# result = iterative_retrieve_adaptive(
#     QUESTION, faiss_index, KNOWLEDGE_BASE, embed_model,
#     max_rounds=4,
# )

### Exercise 3 — Confidence-Based Stopping

Add a fourth convergence criterion: **answer confidence**.  After each round, ask the LLM:

> *"Based on the retrieved passages, can you confidently and fully answer the question?  Reply YES or NO."*

If the LLM replies YES, stop iterating.  Integrate this into the `has_converged` function and test on a question where the answer is available after just one round (e.g., *"How fast does light travel?"*).

## 🔑 Key Takeaways

### What We Built

- A **20-passage knowledge base** designed for multi-hop reasoning
- A **FAISS + sentence-transformers** retrieval layer
- An **LLM-powered query refinement** module using Qwen3-8B
- A **convergence-aware iterative retrieval loop** that knows when to stop
- A **side-by-side comparison** framework for evaluating retrieval strategies

### Core Principles

1. **Single-pass retrieval cannot represent conjunctive information needs.**  A single embedding vector conflates multiple semantic targets, so complex queries always lose information.

2. **Iteration mimics human research behavior.**  The retrieve → read → refine → retrieve loop is the same strategy an expert researcher uses — and it works for the same reasons.

3. **Query refinement is the engine of iteration.**  The quality of the refined query determines how much new information the next round will surface.  LLM-based refinement leverages the model's reasoning to close information gaps efficiently.

4. **Convergence criteria prevent wasted computation.**  Query similarity plateaus, passage novelty checks, and iteration limits ensure the loop terminates gracefully.

5. **Iterative retrieval is the foundation of advanced RAG.**  Techniques like CRAG, Self-RAG, and Agentic RAG all build on the same core loop — they simply add richer decision-making at each step.

### Looking Forward

```
Single-pass RAG ──▶ Iterative Retrieval ──▶ CRAG / Self-RAG ──▶ Agentic RAG
       │                    │                      │                   │
   one shot          multi-turn loop        + self-critique      + tool use
```

In the next notebooks in this course we explore how adding **corrective evaluation** (CRAG) and **self-reflection** (Self-RAG) make the loop smarter by teaching the system to *judge* what it retrieves, not just retrieve more of it.

## 📚 References

- Jiang, Z. et al. (2023). *Active Retrieval Augmented Generation.* arXiv:2305.06983  — Introduces FLARE, an iterative retrieval approach guided by model confidence.
- Yan, S. et al. (2024). *Corrective Retrieval Augmented Generation.* arXiv:2401.15884  — Adds a relevance evaluator to the retrieval loop.
- Asai, A. et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique.* arXiv:2310.11511  — Self-reflection tokens for retrieval and generation.
- Trivedi, H. et al. (2023). *Interleaving Retrieval with Chain-of-Thought Reasoning.* arXiv:2212.10509  — IRCoT: retrieval at each reasoning step.
- Shao, Z. et al. (2023). *Enhancing Retrieval-Augmented Large Language Models with Iterative Retrieval-Generation Synergy.* arXiv:2305.15294